[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abpandey-dev/learn-python-from-scratch-to-advanced/blob/main/notebooks/10-engineering-toolbox/02_testing.ipynb)

*Engineering Toolbox · Notebook 2 of 8*

# Testing With pytest

How testing really works and how to do it well: approximate equality, parametrize ids, fixture order and scopes, mocks, injected clocks, line and branch coverage, mutation testing, flaky tests, property-based shrinking, boundary values and partitions, test selection, the pyramid, snapshots, test smells, summaries and durations.

### 👩‍🏫 From your teacher

**The picture to keep in your head:** Tests are a safety net under a trapeze: you can try bold moves because a mistake gets caught before it hits the ground.

**What interviewers really test here:** Writing pytest tests, fixtures and scopes, mocks at the boundaries, coverage and its limits, flaky tests and the test pyramid.

### How to use this notebook

1. **Run the setup cell** just below.
2. **Read each lesson slowly.** Work the numbers on paper before reading on.
3. **Run and then change the examples.** Breaking code on purpose is how you learn.
4. **Try each practice question before opening an answer.** If you're stuck, open one hint at a time.
5. **Say your interview answers out loud.** Knowing something and explaining it are different skills.

### Your roadmap

- [ ] 1. Your first pytest tests *(Basics)*
- [ ] 2. Fixtures, mocks and testable code *(Core)*
- [ ] 3. How good are your tests? *(Core)*
- [ ] 4. Keeping a test suite fast and trustworthy *(Advanced)*
- [ ] 20 practice questions
- [ ] Interview questions, out loud

In [1]:
# ▶ Run this cell first. It prepares the notebook; you don't need to read it.
import contextlib, copy, io, math, sys
import numpy as np, matplotlib.pyplot as plt
%matplotlib inline
import math
import json
PLAIN = True

class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def mk_list(vals, pos=-1):
    dummy = cur = ListNode()
    nodes = []
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
        nodes.append(cur)
    if pos >= 0 and nodes:
        nodes[-1].next = nodes[pos]
    return dummy.next

def list_arr(head):
    out, seen = [], set()
    while head and id(head) not in seen:
        seen.add(id(head))
        out.append(head.val)
        head = head.next
    return out

def mk_tree(vals):
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    queue, i = [root], 1
    while queue and i < len(vals):
        node = queue.pop(0)
        for side in ("left", "right"):
            if i < len(vals) and vals[i] is not None:
                child = TreeNode(vals[i])
                setattr(node, side, child)
                queue.append(child)
            i += 1
    return root

def tree_arr(root):
    out, queue = [], [root]
    while queue:
        node = queue.pop(0)
        if node:
            out.append(node.val)
            queue += [node.left, node.right]
        else:
            out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out

def find(root, val):
    if not root or root.val == val:
        return root
    return find(root.left, val) or find(root.right, val)

ListNode.__repr__ = lambda self: f"ListNode{list_arr(self)}"
TreeNode.__repr__ = lambda self: f"TreeNode{tree_arr(self)}"

def plain(x, digits=4):
    """NumPy / pandas results as plain Python values, so they compare with a test's expected answer."""
    np = sys.modules.get("numpy")
    pd = sys.modules.get("pandas")
    if pd is not None:
        if isinstance(x, pd.DataFrame):
            return {str(c): plain(x[c].tolist(), digits) for c in x.columns}
        if isinstance(x, pd.Series):
            return {plain(k, digits): plain(v, digits) for k, v in x.items()}
        if isinstance(x, pd.Index):
            return plain(x.tolist(), digits)
        if x is pd.NaT:
            return None
        if isinstance(x, pd.Timestamp):
            return str(x.date()) if x == x.normalize() else str(x)
    if np is not None:
        if isinstance(x, np.ndarray):
            return plain(x.tolist(), digits)
        if isinstance(x, np.generic):
            return plain(x.item(), digits)
    if isinstance(x, float):
        return None if x != x else round(x, digits) + 0.0
    if isinstance(x, list):
        return [plain(v, digits) for v in x]
    if isinstance(x, tuple):
        return tuple(plain(v, digits) for v in x)
    if isinstance(x, dict):
        return {plain(k, digits): plain(v, digits) for k, v in x.items()}
    if isinstance(x, (set, frozenset)):
        return {plain(v, digits) for v in x}
    return x


def check(fn, tests, plain_compare=None, listify=False, design=False, prep=None, post=None,
          norm=None, verify=None, runner=None, inplace=None):
    """Run your code against the tests and show which pass."""
    use_plain = PLAIN if plain_compare is None else plain_compare
    passed, problems = 0, []
    for args, want in tests:
        try:
            a = copy.deepcopy(args)
            if design:
                obj = fn(*a[0])
                got = [getattr(obj, m)(*p) for m, p in a[1]]
            elif runner is not None:
                got = runner(fn, a)
            else:
                if prep is not None:
                    a = prep(a)
                with contextlib.redirect_stdout(io.StringIO()):
                    got = fn(*a)
                if inplace is not None:
                    got = a[inplace]
                if listify:
                    got = list(got)
                if post is not None:
                    got = post(got)
            if verify is not None:
                ok = bool(verify(copy.deepcopy(args), got))
            elif norm is not None:
                ok = norm(copy.deepcopy(got)) == norm(copy.deepcopy(want))
            else:
                ok = (plain(got), plain(want)) if use_plain else (got, want)
                ok = ok[0] == ok[1]
            ok = ok if isinstance(ok, bool) else False
        except Exception as e:  # your code raised an error: report it, don't stop the notebook
            ok, got = False, f"{type(e).__name__}: {e}"
        passed += ok
        if not ok:
            problems.append((args, want, got))
    plt.close("all")
    if passed == len(tests):
        print(f"✅ All {len(tests)} tests passed. Well done! Now compare with the answers below.")
    else:
        print(f"❌ {passed}/{len(tests)} tests passed. First one to fix:")
        args, want, got = problems[0]
        print("   input:   ", repr(args)[:300])
        print("   expected:", repr(want)[:300])
        print("   you got: ", repr(got)[:300])
    return None

print("Ready! Scroll down and start learning.")

Ready! Scroll down and start learning.


---
# Part 1: The theory in one place

Read this once for the big picture. It's fine if it doesn't all stick yet: the lessons will make it concrete, and you can come back to revise.

### Kinds of tests

| Kind | Checks | Speed |
| --- | --- | --- |
| Unit | one function or class, dependencies faked | milliseconds |
| Integration | your code with a real database, queue or service | seconds |
| End-to-end | the whole system through its UI or API | seconds to minutes |
| Contract | that two services still agree on an API | fast |
| Regression | a bug that happened once never returns | any |

### Test doubles

| Double | What it does |
| --- | --- |
| Stub | returns canned answers |
| Mock | records calls so you can assert on them |
| Fake | a working lightweight version (in-memory database) |
| Spy | wraps the real thing and records calls |

---
# Part 2: Lessons, step by step

## Lesson 1: Your first pytest tests

*Level: Basics*

A test is a function whose name starts with `test_`, in a file named `test_*.py`. Inside, **arrange** the inputs, **act** by calling the code, and **assert** the result. pytest rewrites `assert` so failures show both values.



```python
# app/pricing.py
def total(prices, discount=0.0):
    return round(sum(prices) * (1 - discount), 2)

# tests/test_pricing.py
import pytest
from app.pricing import total

def test_total_no_discount():
    assert total([10.0, 5.5]) == 15.5

def test_total_with_discount():
    assert total([10.0, 10.0], discount=0.1) == pytest.approx(18.0)

@pytest.mark.parametrize("prices, expected", [([], 0), ([1], 1), ([0.1, 0.2], 0.3)])
def test_total_cases(prices, expected):
    assert total(prices) == pytest.approx(expected)

def test_negative_discount_rejected():
    with pytest.raises(ValueError):
        total([1], discount=-0.5)
```



```python
pip install pytest
pytest -q                 # run everything
pytest tests/test_pricing.py::test_total_cases -v
```

**✏️ Your turn:** you now have what you need for [Q1](#q1), [Q2](#q2), [Q9](#q9), [Q6](#q6), [Q7](#q7). Try one now, or carry on and do them in Part 3.

## Lesson 2: Fixtures, mocks and testable code

*Level: Core*

**Fixtures** provide what tests need (a temporary folder, a database, an app client) and clean up afterwards; `yield` splits setup from teardown. Pick the scope by cost: `function` (fresh each test), `module`, or `session`.



```python
@pytest.fixture(scope="module")
def db():
    conn = sqlite3.connect(":memory:")
    conn.execute("CREATE TABLE users (id INTEGER, name TEXT)")
    yield conn            # tests run here
    conn.close()          # teardown

def test_insert(db, tmp_path):            # tmp_path is a built-in fixture
    db.execute("INSERT INTO users VALUES (1, 'Asha')")
    assert db.execute("SELECT COUNT(*) FROM users").fetchone()[0] == 1
```



Replace slow or external things (email, payments, HTTP, the clock) with **test doubles**. The easiest code to test takes its dependencies as arguments (**dependency injection**) instead of creating them inside.



```python
from unittest.mock import Mock
def test_reminders_sent():
    send = Mock()
    remind_overdue(invoices, today=20, send_email=send)
    send.assert_called_once_with("a@x.com", "Invoice 1 is overdue")

def test_api(monkeypatch):                 # or patch where it's used
    monkeypatch.setattr("app.client.requests.get", lambda url, timeout: FakeResponse(200))
```

**✏️ Your turn:** you now have what you need for [Q11](#q11), [Q12](#q12), [Q13](#q13), [Q3](#q3). Try one now, or carry on and do them in Part 3.

## Lesson 3: How good are your tests?

*Level: Core*

- **Line coverage** shows code never run by any test. **Branch coverage** also checks both sides of each `if`. Use coverage to find gaps — 100% doesn't mean correct.
- **Mutation testing** (mutmut) changes your code in small ways; a surviving mutant is a bug your tests wouldn't catch.
- **Property-based testing** (Hypothesis) generates many inputs and shrinks a failure to the smallest example.
- **Snapshot tests** catch unexpected changes in large outputs — review snapshot updates carefully.



```python
pip install pytest-cov hypothesis
pytest --cov=app --cov-branch --cov-report=term-missing

from hypothesis import given, strategies as st
@given(st.lists(st.integers()))
def test_sort_is_idempotent(xs):
    assert sorted(sorted(xs)) == sorted(xs)
```

**✏️ Your turn:** you now have what you need for [Q14](#q14), [Q4](#q4), [Q15](#q15), [Q20](#q20), [Q17](#q17). Try one now, or carry on and do them in Part 3.

## Lesson 4: Keeping a test suite fast and trustworthy

*Level: Advanced*

- Follow the **test pyramid**: many fast unit tests, fewer integration tests, a handful of end-to-end tests.
- **Flaky tests** destroy trust. Find them from history, quarantine them, and fix the cause (time, ordering, shared state, network) — retries only hide it.
- In big repos, run only the tests **affected** by a change on each push, and everything nightly.
- Watch the slowest tests (`pytest --durations=10`) and run in parallel with `pytest -n auto` (pytest-xdist).
- Avoid test smells: no asserts, sleeps, prints, tests that depend on each other's order.

**✏️ Your turn:** you now have what you need for [Q5](#q5), [Q19](#q19), [Q16](#q16), [Q8](#q8), [Q10](#q10), [Q18](#q18). Try one now, or carry on and do them in Part 3.

---
# Part 3: Practice like an interview

There are 20 questions here, easiest first.

**Work the way you would in a real interview:** restate the problem, try an example by hand, say the simple approach and its cost, improve it, then write the code. Test it with the cell. Only open an answer once you've made a real attempt, because struggling is where the learning happens.

**▶ Run this cell before the questions.** It sets up the data and tools they share.

In [2]:
import itertools

from collections import Counter, defaultdict


<a id="q1"></a>
### Q1. Compare Floats Like pytest.approx

🟢 Warm-up · *Never use == on floats*

`pytest.approx` treats two numbers as equal if they differ by at most `max(rel × |expected|, abs)` with rel = 1e-6 and abs = 1e-12. Return a bool for each `(got, expected)` pair.

**Examples**

- `pairs = [(0.30000000000000004, 0.3), (1e-13, 0.0), (100.00005, 100.0), (100.001, 100.0), (1000000.5, 1000000.0), (2.5, 2.6)]` → `[True, True, True, False, True, False]`

In [ ]:
def approx_equal(pairs):
    """
    pytest.approx treats two numbers as equal if they differ by at most max(rel
    × |expected|, abs) with rel = 1e-6 and abs = 1e-12. Return a bool for each
    (got, expected) pair.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(approx_equal, [
    (([(0.30000000000000004, 0.3), (1e-13, 0.0), (100.00005, 100.0), (100.001, 100.0), (1000000.5, 1000000.0), (2.5, 2.6)],), [True, True, True, False, True, False]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Compute the tolerance by hand.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

One comprehension. (Note `math.isclose` is slightly different: its relative tolerance uses the larger of the two numbers, while `approx` uses the expected value.)

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Compute the tolerance by hand.

```python
def approx_equal(pairs):
    out = []
    for got, want in pairs:
        tol = max(1e-6 * abs(want), 1e-12)
        out.append(abs(got - want) <= tol)
    return out
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** One comprehension. (Note `math.isclose` is slightly different: its relative tolerance uses the larger of the two numbers, while `approx` uses the expected value.)

```python
def approx_equal(pairs):
    return [abs(g - w) <= max(1e-6 * abs(w), 1e-12) for g, w in pairs]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Compute the tolerance by hand. That costs O(n) time and O(n) space."

"I can do better: One comprehension. (Note `math.isclose` is slightly different: its relative tolerance uses the larger of the two numbers, while `approx` uses the expected value.) That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/testing/compare-floats-like-pytest-approx)

<a id="q2"></a>
### Q2. Parametrize: Generate Test Ids

🟢 Warm-up · *One test, many cases*

`@pytest.mark.parametrize` stacked on several arguments runs the cartesian product. Given the test name and an ordered list of `(argument, values)`, return the test ids pytest shows, e.g. `test_price[card-1]` (values joined by `-`, first argument varying slowest).

**Examples**

- `name = 'test_price', params = [('method', ['card', 'upi']), ('qty', [1, 10]), ('member', [True, False])]` → `['test_price[card-1-True]', 'test_price[card-1-False]', 'test_price[card-10-True]', 'test_price[card-10-False]', 'test_price[upi-1-True]', 'test_price[upi-1-False]', 'test_price[upi-10-True]', 'test_price[upi-10-False]']`

In [ ]:
def param_ids(name, params):
    """
    @pytest.mark.parametrize stacked on several arguments runs the cartesian
    product. Given the test name and an ordered list of (argument, values),
    return the test ids pytest shows, e.g. test_price[card-1] (values joined by
    -, first argument varying slowest).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(param_ids, [
    (('test_price', [('method', ['card', 'upi']), ('qty', [1, 10]), ('member', [True, False])]), ['test_price[card-1-True]', 'test_price[card-1-False]', 'test_price[card-10-True]', 'test_price[card-10-False]', 'test_price[upi-1-True]', 'test_price[upi-1-False]', 'test_price[upi-10-True]', 'test_price[upi-10-False]']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Recursive nested loops.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`itertools.product`.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Recursive nested loops.

```python
def param_ids(name, params):
    combos = [[]]
    for _, values in params:
        new = []
        for c in combos:
            for v in values:
                new.append(c + [str(v)])
        combos = new
    return [f"{name}[{'-'.join(c)}]" for c in combos]
```

**Cost:** O(product) time, O(product) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `itertools.product`.

```python
def param_ids(name, params):
    return [f"{name}[{'-'.join(map(str, c))}]" for c in itertools.product(*(v for _, v in params))]
```

**Cost:** O(product) time, O(product) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Recursive nested loops. That costs O(product) time and O(product) space."

"I can do better: `itertools.product`. That's O(product) time and O(product) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/testing/parametrize-generate-test-ids)

<a id="q3"></a>
### Q3. Inject the Clock

🟢 Warm-up · *Make time testable*

Code that calls `time.time()` inside is hard to test. Pass `now` in instead. Given sessions `(id, started_at, ttl)` and a list of 'now' values, return the expired session ids at each moment (expired when now ≥ start + ttl).

**Examples**

- `sessions = [('s1', 0, 30), ('s2', 10, 30), ('s3', 25, 5), ('s4', 40, 60)], moments = [20, 30, 45, 120]` → `[[], ['s1', 's3'], ['s1', 's2', 's3'], ['s1', 's2', 's3', 's4']]`

In [ ]:
def expired_at(sessions, moments):
    """
    Code that calls time.time() inside is hard to test. Pass now in instead.
    Given sessions (id, started_at, ttl) and a list of 'now' values, return the
    expired session ids at each moment (expired when now ≥ start + ttl).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(expired_at, [
    (([('s1', 0, 30), ('s2', 10, 30), ('s3', 25, 5), ('s4', 40, 60)], [20, 30, 45, 120]), [[], ['s1', 's3'], ['s1', 's2', 's3'], ['s1', 's2', 's3', 's4']]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Nested loops.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A comprehension per moment.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Nested loops.

```python
def expired_at(sessions, moments):
    out = []
    for now in moments:
        gone = []
        for sid, start, ttl in sessions:
            if now >= start + ttl:
                gone.append(sid)
        out.append(gone)
    return out
```

**Cost:** O(t · s) time, O(s) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A comprehension per moment.

```python
def expired_at(sessions, moments):
    return [[s for s, start, ttl in sessions if now >= start + ttl] for now in moments]
```

**Cost:** O(t · s) time, O(s) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Nested loops. That costs O(t · s) time and O(s) space."

"I can do better: A comprehension per moment. That's O(t · s) time and O(s) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/testing/inject-the-clock)

<a id="q4"></a>
### Q4. Branch Coverage

🟢 Warm-up · *Did both sides of each if run?*

Each decision point (an `if` line) has two outcomes. `taken` maps a decision line to the outcomes seen in the tests (`'T'` and/or `'F'`). Return `(branch coverage % rounded to 1, decisions with a missing outcome as (line, missing))`. 100% line coverage can still miss branches.

**Examples**

- `taken = {12: ['T', 'F'], 18: ['T'], 25: ['F'], 31: ['T', 'F'], 40: []}` → `(60.0, [(18, ['F']), (25, ['T']), (40, ['T', 'F'])])`

In [ ]:
def branch_coverage(taken):
    """
    Each decision point (an if line) has two outcomes. taken maps a decision
    line to the outcomes seen in the tests ('T' and/or 'F'). Return (branch
    coverage % rounded to 1, decisions with a missing outcome as (line,
    missing)). 100% line coverage can still miss branches.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(branch_coverage, [
    (({12: ['T', 'F'], 18: ['T'], 25: ['F'], 31: ['T', 'F'], 40: []},), (60.0, [(18, ['F']), (25, ['T']), (40, ['T', 'F'])])),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Count outcomes in a loop.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Set differences.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Count outcomes in a loop.

```python
def branch_coverage(taken):
    total, hit, gaps = 0, 0, []
    for line, outs in taken.items():
        total += 2
        hit += len(set(outs))
        miss = [o for o in ["T", "F"] if o not in outs]
        if miss:
            gaps.append((line, miss))
    return (round(hit / total * 100, 1), gaps)
```

**Cost:** O(d) time, O(d) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Set differences.

```python
def branch_coverage(taken):
    gaps = [(l, sorted({"T", "F"} - set(o), reverse=True)) for l, o in taken.items() if {"T", "F"} - set(o)]
    return (round(100 * sum(len(set(o)) for o in taken.values()) / (2 * len(taken)), 1), gaps)
```

**Cost:** O(d) time, O(d) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Count outcomes in a loop. That costs O(d) time and O(d) space."

"I can do better: Set differences. That's O(d) time and O(d) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/testing/branch-coverage)

<a id="q5"></a>
### Q5. Find the Flaky Tests

🟢 Warm-up · *Same code, different result*

Runs are `(commit, test, outcome)`. A test is flaky if it both passed and failed on the **same** commit. Return the flaky tests (sorted) and, for each, the fraction of its runs that failed (rounded to 2).

**Examples**

- `runs = [('c1', 'test_login', 'pass'), ('c1', 'test_login', 'fail'), ('c1', 'test_cart', 'pass'), ('c2', 'test_cart', 'fail'), ('c2', 'test_cart', 'fail'), ('c2', 'test_search', 'pass'), ('c2', 'test_search', 'fail'), ('c3', 'test_search', 'pass'), ('c3', 'test_login', 'pass')]` → `[('test_login', 0.33), ('test_search', 0.33)]`

In [ ]:
def flaky_tests(runs):
    """
    Runs are (commit, test, outcome). A test is flaky if it both passed and
    failed on the same commit. Return the flaky tests (sorted) and, for each,
    the fraction of its runs that failed (rounded to 2).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(flaky_tests, [
    (([('c1', 'test_login', 'pass'), ('c1', 'test_login', 'fail'), ('c1', 'test_cart', 'pass'), ('c2', 'test_cart', 'fail'), ('c2', 'test_cart', 'fail'), ('c2', 'test_search', 'pass'), ('c2', 'test_search', 'fail'), ('c3', 'test_search', 'pass'), ('c3', 'test_login', 'pass')],), [('test_login', 0.33), ('test_search', 0.33)]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Nested dicts of outcome sets.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`defaultdict(set)` keyed by (commit, test).

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Nested dicts of outcome sets.

```python
def flaky_tests(runs):
    outcomes, totals, fails = {}, {}, {}
    for c, t, o in runs:
        outcomes.setdefault((c, t), set()).add(o)
        totals[t] = totals.get(t, 0) + 1
        fails[t] = fails.get(t, 0) + (o == "fail")
    flaky = sorted({t for (c, t), s in outcomes.items() if len(s) == 2})
    return [(t, round(fails[t] / totals[t], 2)) for t in flaky]
```

**Cost:** O(r) time, O(r) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `defaultdict(set)` keyed by (commit, test).

```python
def flaky_tests(runs):
    seen, per = defaultdict(set), defaultdict(list)
    for c, t, o in runs:
        seen[c, t].add(o)
        per[t].append(o == "fail")
    return [(t, round(sum(per[t]) / len(per[t]), 2)) for t in sorted({t for (_, t), s in seen.items() if len(s) == 2})]
```

**Cost:** O(r) time, O(r) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Nested dicts of outcome sets. That costs O(r) time and O(r) space."

"I can do better: `defaultdict(set)` keyed by (commit, test). That's O(r) time and O(r) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/testing/find-the-flaky-tests)

<a id="q6"></a>
### Q6. Boundary Values

🟢 Warm-up · *Bugs live at the edges*

For each valid range `(lo, hi)` (inclusive), return the boundary test inputs: lo − 1, lo, lo + 1, hi − 1, hi, hi + 1 (sorted, without duplicates).

**Examples**

- `ranges = [(18, 65), (1, 1), (0, 2)]` → `[[17, 18, 19, 64, 65, 66], [0, 1, 2], [-1, 0, 1, 2, 3]]`

In [ ]:
def boundaries(ranges):
    """
    For each valid range (lo, hi) (inclusive), return the boundary test inputs:
    lo − 1, lo, lo + 1, hi − 1, hi, hi + 1 (sorted, without duplicates).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(boundaries, [
    (([(18, 65), (1, 1), (0, 2)],), [[17, 18, 19, 64, 65, 66], [0, 1, 2], [-1, 0, 1, 2, 3]]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Build a list and dedupe by hand.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A sorted set per range.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Build a list and dedupe by hand.

```python
def boundaries(ranges):
    out = []
    for lo, hi in ranges:
        vals = []
        for v in [lo - 1, lo, lo + 1, hi - 1, hi, hi + 1]:
            if v not in vals:
                vals.append(v)
        out.append(sorted(vals))
    return out
```

**Cost:** O(r) time, O(r) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A sorted set per range.

```python
def boundaries(ranges):
    return [sorted({lo - 1, lo, lo + 1, hi - 1, hi, hi + 1}) for lo, hi in ranges]
```

**Cost:** O(r) time, O(r) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Build a list and dedupe by hand. That costs O(r) time and O(r) space."

"I can do better: A sorted set per range. That's O(r) time and O(r) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/testing/boundary-values)

<a id="q7"></a>
### Q7. Equivalence Partitions

🟢 Warm-up · *One test per kind of input*

Inputs that behave the same belong to one partition; one test per partition is enough (plus boundaries). Using `shipping_band`, return which bands the given test weights cover and which bands are missing (from `invalid, letter, parcel, heavy, freight`).

**Examples**

- `weights = [0.5, 2, 3, 4.9, 12]` → `(['letter', 'parcel', 'heavy'], ['invalid', 'freight'])`

In [ ]:
def partitions(weights):
    """
    Inputs that behave the same belong to one partition; one test per partition
    is enough (plus boundaries). Using shipping_band, return which bands the
    given test weights cover and which bands are missing (from invalid, letter,
    parcel, heavy, freight).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(partitions, [
    (([0.5, 2, 3, 4.9, 12],), (['letter', 'parcel', 'heavy'], ['invalid', 'freight'])),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop and collect into a list.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Set operations.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Loop and collect into a list.

```python
def shipping_band(kg):
    if kg <= 0:
        return "invalid"
    if kg <= 1:
        return "letter"
    if kg <= 5:
        return "parcel"
    if kg <= 30:
        return "heavy"
    return "freight"


def partitions(weights):
    bands = ["invalid", "letter", "parcel", "heavy", "freight"]
    covered = []
    for w in weights:
        b = shipping_band(w)
        if b not in covered:
            covered.append(b)
    return ([b for b in bands if b in covered], [b for b in bands if b not in covered])
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Set operations.

```python
def shipping_band(kg):
    if kg <= 0:
        return "invalid"
    if kg <= 1:
        return "letter"
    if kg <= 5:
        return "parcel"
    if kg <= 30:
        return "heavy"
    return "freight"


def partitions(weights):
    bands = ["invalid", "letter", "parcel", "heavy", "freight"]
    got = set(map(shipping_band, weights))
    return ([b for b in bands if b in got], [b for b in bands if b not in got])
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop and collect into a list. That costs O(n) time and O(1) space."

"I can do better: Set operations. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/testing/equivalence-partitions)

<a id="q8"></a>
### Q8. Check the Test Pyramid

🟢 Warm-up · *Many fast tests, few slow ones*

Given counts and average seconds for `unit`, `integration` and `e2e` tests, return `(total minutes rounded to 1, % of tests that are unit rounded to 1, shape)` where shape is `'pyramid'` if unit &gt; integration &gt; e2e, `'ice cream cone'` if e2e is the largest group, else `'diamond'`.

**Examples**

- `suite = {'unit': (1200, 0.02), 'integration': (150, 0.8), 'e2e': (20, 12.0)}` → `(6.4, 87.6, 'pyramid')`
- `suite = {'unit': (40, 0.05), 'integration': (60, 1.5), 'e2e': (300, 20.0)}` → `(101.5, 10.0, 'ice cream cone')`
- `suite = {'unit': (100, 0.02), 'integration': (400, 0.5), 'e2e': (30, 9.0)}` → `(7.9, 18.9, 'diamond')`

In [ ]:
def pyramid(suite):
    """
    Given counts and average seconds for unit, integration and e2e tests, return
    (total minutes rounded to 1, % of tests that are unit rounded to 1, shape)
    where shape is 'pyramid' if unit > integration > e2e, 'ice cream cone' if
    e2e is the largest group, else 'diamond'.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(pyramid, [
    (({'unit': (1200, 0.02), 'integration': (150, 0.8), 'e2e': (20, 12.0)},), (6.4, 87.6, 'pyramid')),
    (({'unit': (40, 0.05), 'integration': (60, 1.5), 'e2e': (300, 20.0)},), (101.5, 10.0, 'ice cream cone')),
    (({'unit': (100, 0.02), 'integration': (400, 0.5), 'e2e': (30, 9.0)},), (7.9, 18.9, 'diamond')),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Step-by-step arithmetic and if/elif.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Same, compact.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Step-by-step arithmetic and if/elif.

```python
def pyramid(suite):
    u, i, e = suite["unit"][0], suite["integration"][0], suite["e2e"][0]
    seconds = 0
    for n, t in suite.values():
        seconds += n * t
    if u > i > e:
        shape = "pyramid"
    elif e >= u and e >= i:
        shape = "ice cream cone"
    else:
        shape = "diamond"
    return (round(seconds / 60, 1), round(u / (u + i + e) * 100, 1), shape)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Same, compact.

```python
def pyramid(suite):
    u, i, e = (suite[k][0] for k in ("unit", "integration", "e2e"))
    shape = "pyramid" if u > i > e else "ice cream cone" if e >= max(u, i) else "diamond"
    return (round(sum(n * t for n, t in suite.values()) / 60, 1), round(100 * u / (u + i + e), 1), shape)
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Step-by-step arithmetic and if/elif. That costs O(1) time and O(1) space."

"I can do better: Same, compact. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/testing/check-the-test-pyramid)

<a id="q9"></a>
### Q9. Write the Summary Line

🟢 Warm-up · *What pytest prints at the end*

Given test outcomes and the total seconds, return pytest's summary line, e.g. `'1 failed, 12 passed, 2 skipped in 3.42s'`: counts in the order failed, passed, skipped, xfailed, xpassed, error (pluralised to `errors` when more than one), omitting zeros, seconds to 2 decimals.

**Examples**

- `outcomes = ['passed', 'passed', 'passed', 'passed', 'passed', 'passed', 'passed', 'passed', 'passed', 'passed', 'passed', 'passed', 'failed', 'skipped', 'skipped'], seconds = 3.4219` → `'1 failed, 12 passed, 2 skipped in 3.42s'`
- `outcomes = ['passed', 'error', 'error', 'xfailed'], seconds = 0.5` → `'1 passed, 1 xfailed, 2 errors in 0.50s'`

In [ ]:
def summary_line(outcomes, seconds):
    """
    Given test outcomes and the total seconds, return pytest's summary line,
    e.g. '1 failed, 12 passed, 2 skipped in 3.42s': counts in the order failed,
    passed, skipped, xfailed, xpassed, error (pluralised to errors when more
    than one), omitting zeros, seconds to 2 decimals.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(summary_line, [
    ((['passed', 'passed', 'passed', 'passed', 'passed', 'passed', 'passed', 'passed', 'passed', 'passed', 'passed', 'passed', 'failed', 'skipped', 'skipped'], 3.4219), '1 failed, 12 passed, 2 skipped in 3.42s'),
    ((['passed', 'error', 'error', 'xfailed'], 0.5), '1 passed, 1 xfailed, 2 errors in 0.50s'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Count each outcome with a loop.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`Counter` and a join.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Count each outcome with a loop.

```python
def summary_line(outcomes, seconds):
    order = ["failed", "passed", "skipped", "xfailed", "xpassed", "error"]
    parts = []
    for o in order:
        n = 0
        for x in outcomes:
            if x == o:
                n += 1
        if n:
            word = "errors" if o == "error" and n > 1 else o
            parts.append(f"{n} {word}")
    return ", ".join(parts) + f" in {seconds:.2f}s"
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `Counter` and a join.

```python
def summary_line(outcomes, seconds):
    c = Counter(outcomes)
    word = lambda o: "errors" if o == "error" and c[o] > 1 else o
    return ", ".join(f"{c[o]} {word(o)}" for o in ["failed", "passed", "skipped", "xfailed", "xpassed", "error"] if c[o]) + f" in {seconds:.2f}s"
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Count each outcome with a loop. That costs O(n) time and O(1) space."

"I can do better: `Counter` and a join. That's O(n) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/testing/write-the-summary-line)

<a id="q10"></a>
### Q10. Find the Slowest Tests

🟢 Warm-up · *pytest --durations=3*

Given `(test, seconds)`, return the `k` slowest tests (ties by name) and the share of total time they take (% rounded to 1).

**Examples**

- `durations = [('test_login', 0.12), ('test_export_pdf', 8.4), ('test_search', 0.9), ('test_db_migration', 14.2), ('test_cart', 0.05), ('test_email', 3.1)], k = 3` → `([('test_db_migration', 14.2), ('test_export_pdf', 8.4), ('test_email', 3.1)], 96.0)`

In [ ]:
def slowest(durations, k):
    """
    Given (test, seconds), return the k slowest tests (ties by name) and the
    share of total time they take (% rounded to 1).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(slowest, [
    (([('test_login', 0.12), ('test_export_pdf', 8.4), ('test_search', 0.9), ('test_db_migration', 14.2), ('test_cart', 0.05), ('test_email', 3.1)], 3), ([('test_db_migration', 14.2), ('test_export_pdf', 8.4), ('test_email', 3.1)], 96.0)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Sort everything.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`heapq.nlargest`.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Sort everything.

```python
def slowest(durations, k):
    top = sorted(durations, key=lambda d: (-d[1], d[0]))[:k]
    total = 0
    for _, s in durations:
        total += s
    share = 0
    for _, s in top:
        share += s
    return (top, round(share / total * 100, 1))
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `heapq.nlargest`.

```python
def slowest(durations, k):
    import heapq
    top = heapq.nsmallest(k, durations, key=lambda d: (-d[1], d[0]))
    return (top, round(100 * sum(s for _, s in top) / sum(s for _, s in durations), 1))
```

**Cost:** O(n log k) time, O(k) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Sort everything. That costs O(n log n) time and O(n) space."

"I can do better: `heapq.nlargest`. That's O(n log k) time and O(k) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/testing/find-the-slowest-tests)

<a id="q11"></a>
### Q11. Fixture Setup and Teardown Order

🟡 Interview standard · *Dependencies first, cleanup in reverse*

Fixtures can request other fixtures (`deps`). For a test requesting `wanted` fixtures in order, pytest sets each up after its own dependencies (depth-first, in request order, each only once) and tears them down in exact reverse order. Return `(setup order, teardown order)`.

**Examples**

- `deps = {'db': ['config'], 'client': ['app', 'db'], 'app': ['config'], 'config': [], 'tmp_dir': []}, wanted = ['client', 'tmp_dir', 'db']` → `(['config', 'app', 'db', 'client', 'tmp_dir'], ['tmp_dir', 'client', 'db', 'app', 'config'])`

In [ ]:
def fixture_order(deps, wanted):
    """
    Fixtures can request other fixtures (deps). For a test requesting wanted
    fixtures in order, pytest sets each up after its own dependencies (depth-
    first, in request order, each only once) and tears them down in exact
    reverse order. Return (setup order, teardown order).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(fixture_order, [
    (({'db': ['config'], 'client': ['app', 'db'], 'app': ['config'], 'config': [], 'tmp_dir': []}, ['client', 'tmp_dir', 'db']), (['config', 'app', 'db', 'client', 'tmp_dir'], ['tmp_dir', 'client', 'db', 'app', 'config'])),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Recursive depth-first setup with a visited list.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Same, with a set for visited.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Recursive depth-first setup with a visited list.

```python
def fixture_order(deps, wanted):
    order = []
    def setup(f):
        if f in order:
            return
        for d in deps[f]:
            setup(d)
        order.append(f)
    for f in wanted:
        setup(f)
    return (order, list(reversed(order)))
```

**Cost:** O(f + d) time, O(f) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Same, with a set for visited.

```python
def fixture_order(deps, wanted):
    order, seen = [], set()
    def setup(f):
        if f not in seen:
            seen.add(f)
            for d in deps[f]:
                setup(d)
            order.append(f)
    for f in wanted:
        setup(f)
    return (order, order[::-1])
```

**Cost:** O(f + d) time, O(f) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Recursive depth-first setup with a visited list. That costs O(f + d) time and O(f) space."

"I can do better: Same, with a set for visited. That's O(f + d) time and O(f) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `deps = {'a': ['b'], 'b': []}, wanted = ['a']`

1. setup() returns None
2. setup() returns None
3. fixture\_order() returns (['b', 'a'], ['a', 'b'])

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/testing/fixture-setup-and-teardown-order)

<a id="q12"></a>
### Q12. How Often Does Each Fixture Run?

🟡 Interview standard · *Fixture scopes*

Tests are `(module, [fixtures used])`. A `function`-scoped fixture is set up for every test that uses it, a `module`-scoped one once per module that uses it, a `session`-scoped one once in total. Return set-up counts per fixture, sorted by name. Expensive fixtures (a database) should have wide scopes.

**Examples**

- `tests = [('test_api.py', ['client', 'db']), ('test_api.py', ['client']), ('test_api.py', ['db', 'tmp_path']), ('test_models.py', ['db']), ('test_models.py', ['tmp_path'])], scopes = {'client': 'function', 'db': 'module', 'tmp_path': 'function', 'settings': 'session'}` → `[('client', 2), ('db', 2), ('tmp_path', 2)]`

In [ ]:
def fixture_setups(tests, scopes):
    """
    Tests are (module, [fixtures used]). A function-scoped fixture is set up for
    every test that uses it, a module-scoped one once per module that uses it, a
    session-scoped one once in total. Return set-up counts per fixture, sorted
    by name. Expensive fixtures (a database) should have wide scopes.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(fixture_setups, [
    (([('test_api.py', ['client', 'db']), ('test_api.py', ['client']), ('test_api.py', ['db', 'tmp_path']), ('test_models.py', ['db']), ('test_models.py', ['tmp_path'])], {'client': 'function', 'db': 'module', 'tmp_path': 'function', 'settings': 'session'}), [('client', 2), ('db', 2), ('tmp_path', 2)]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Track what's alive per scope while looping.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Count distinct scope keys with sets.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Track what's alive per scope while looping.

```python
def fixture_setups(tests, scopes):
    counts, seen = {}, set()
    for i, (module, fixtures) in enumerate(tests):
        for f in fixtures:
            key = {"function": (f, i), "module": (f, module), "session": (f,)}[scopes[f]]
            if key not in seen:
                seen.add(key)
                counts[f] = counts.get(f, 0) + 1
    return sorted(counts.items())
```

**Cost:** O(t · f) time, O(f) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Count distinct scope keys with sets.

```python
def fixture_setups(tests, scopes):
    keys = defaultdict(set)
    for i, (module, fixtures) in enumerate(tests):
        for f in fixtures:
            keys[f].add({"function": i, "module": module, "session": 0}[scopes[f]])
    return sorted((f, len(k)) for f, k in keys.items())
```

**Cost:** O(t · f) time, O(t · f) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Track what's alive per scope while looping. That costs O(t · f) time and O(f) space."

"I can do better: Count distinct scope keys with sets. That's O(t · f) time and O(t · f) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/testing/how-often-does-each-fixture-run)

<a id="q13"></a>
### Q13. Mock a Dependency and Check the Calls

🟡 Interview standard · *Test behaviour without sending real email*

`remind_overdue(invoices, today, send_email)` emails customers with unpaid invoices past due. Call it with a test double for `send_email` and return `(number of calls, list of (to, subject))`. The brute force records calls in a hand-written fake; the optimal uses `unittest.mock.Mock`.

**Examples**

- `invoices = [{'id': 1, 'email': 'a@x.com', 'due': 10, 'paid': False}, {'id': 2, 'email': 'b@x.com', 'due': 30, 'paid': False}, {'id': 3, 'email': 'c@x.com', 'due': 5, 'paid': True}, {'id': 4, 'email': 'd@x.com', 'due': 19, 'paid': False}], today = 20` → `(2, [('a@x.com', 'Invoice 1 is overdue'), ('d@x.com', 'Invoice 4 is overdue')])`

In [ ]:
from unittest.mock import Mock

def mock_calls(invoices, today):
    """
    remind_overdue(invoices, today, send_email) emails customers with unpaid
    invoices past due. Call it with a test double for send_email and return
    (number of calls, list of (to, subject)). The brute force records calls in a
    hand-written fake; the optimal uses unittest.mock.Mock.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(mock_calls, [
    (([{'id': 1, 'email': 'a@x.com', 'due': 10, 'paid': False}, {'id': 2, 'email': 'b@x.com', 'due': 30, 'paid': False}, {'id': 3, 'email': 'c@x.com', 'due': 5, 'paid': True}, {'id': 4, 'email': 'd@x.com', 'due': 19, 'paid': False}], 20), (2, [('a@x.com', 'Invoice 1 is overdue'), ('d@x.com', 'Invoice 4 is overdue')])),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A list-appending fake function.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`Mock()` and `call_args_list`.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** A list-appending fake function.

```python
def remind_overdue(invoices, today, send_email):
    for inv in invoices:
        if not inv["paid"] and inv["due"] < today:
            send_email(inv["email"], f"Invoice {inv['id']} is overdue")


def mock_calls(invoices, today):
    calls = []
    def fake_send(to, subject):
        calls.append((to, subject))
    remind_overdue(invoices, today, fake_send)
    return (len(calls), calls)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `Mock()` and `call_args_list`.

```python
def remind_overdue(invoices, today, send_email):
    for inv in invoices:
        if not inv["paid"] and inv["due"] < today:
            send_email(inv["email"], f"Invoice {inv['id']} is overdue")


def mock_calls(invoices, today):
    send = Mock()
    remind_overdue(invoices, today, send)
    return (send.call_count, [c.args for c in send.call_args_list])
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A list-appending fake function. That costs O(n) time and O(n) space."

"I can do better: `Mock()` and `call_args_list`. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/testing/mock-a-dependency-and-check-the-calls)

<a id="q14"></a>
### Q14. Line Coverage and Missing Lines

🟡 Interview standard · *pytest --cov --cov-report=term-missing*

Given the executable line numbers of a file and the lines each test executed, return `(coverage % rounded to 1, missing lines as ranges)`, written like `'12-14'` or `'20'`. A range runs over consecutive **executable** lines — blank lines and comments in between don't break it, as in coverage.py.

**Examples**

- `executable = [1, 2, 3, 5, 6, 7, 8, 10, 11, 12, 14, 15, 16, 20], runs = [[1, 2, 3, 5, 6], [1, 2, 3, 10, 11, 14], [1, 2, 7, 8]]` → `(71.4, ['12', '15-20'])`

In [ ]:
def coverage(executable, runs):
    """
    Given the executable line numbers of a file and the lines each test
    executed, return (coverage % rounded to 1, missing lines as ranges), written
    like '12-14' or '20'. A range runs over consecutive executable lines — blank
    lines and comments in between don't break it, as in coverage.py.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(coverage, [
    (([1, 2, 3, 5, 6, 7, 8, 10, 11, 12, 14, 15, 16, 20], [[1, 2, 3, 5, 6], [1, 2, 3, 10, 11, 14], [1, 2, 7, 8]]), (71.4, ['12', '15-20'])),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Union in a loop, then walk the missing lines.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Set union and grouping consecutive runs.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Union in a loop, then walk the missing lines.

```python
def coverage(executable, runs):
    hit = []
    for r in runs:
        for line in r:
            if line not in hit:
                hit.append(line)
    missing = [l for l in executable if l not in hit]
    ranges, i = [], 0
    while i < len(missing):
        j = i
        while j + 1 < len(missing) and executable.index(missing[j + 1]) == executable.index(missing[j]) + 1:
            j += 1
        ranges.append(str(missing[i]) if i == j else f"{missing[i]}-{missing[j]}")
        i = j + 1
    covered = len(executable) - len(missing)
    return (round(covered / len(executable) * 100, 1), ranges)
```

**Cost:** O(l + e) time, O(l) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Set union and grouping consecutive runs.

```python
def coverage(executable, runs):
    hit = set().union(*runs)
    pos = {l: i for i, l in enumerate(executable)}
    missing = [l for l in executable if l not in hit]
    ranges = []
    for _, grp in itertools.groupby(missing, key=lambda l: pos[l] - missing.index(l)):
        g = list(grp)
        ranges.append(str(g[0]) if len(g) == 1 else f"{g[0]}-{g[-1]}")
    return (round(100 * (1 - len(missing) / len(executable)), 1), ranges)
```

**Cost:** O(l log l) time, O(l) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Union in a loop, then walk the missing lines. That costs O(l + e) time and O(l) space."

"I can do better: Set union and grouping consecutive runs. That's O(l log l) time and O(l) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/testing/line-coverage-and-missing-lines)

<a id="q15"></a>
### Q15. Mutation Testing: Are Your Tests Any Good?

🟡 Interview standard · *Tests that catch small bugs*

Mutation testing makes small changes (mutants) to the code and checks the tests catch them. The function is `original`, an expression in `age`; each mutant is another expression; tests are `(age, expected)`. A mutant is **killed** if any test's expected value differs from the mutant's result. Return `(killed mutants, survivors, mutation score %)` — survivors point at missing tests.

**Examples**

- `original = 'age >= 18', mutants = ['age > 18', 'age <= 18', 'age >= 17', 'age >= 19', 'True'], tests = [(30, True), (5, False)]` → `(['age <= 18', 'True'], ['age > 18', 'age >= 17', 'age >= 19'], 40.0)`
- `original = 'age >= 18', mutants = ['age > 18', 'age <= 18', 'age >= 17', 'age >= 19', 'True'], tests = [(30, True), (5, False), (18, True), (17, False)]` → `(['age > 18', 'age <= 18', 'age >= 17', 'age >= 19', 'True'], [], 100.0)`

In [ ]:
def mutation_score(original, mutants, tests):
    """
    Mutation testing makes small changes (mutants) to the code and checks the
    tests catch them. The function is original, an expression in age; each
    mutant is another expression; tests are (age, expected). A mutant is killed
    if any test's expected value differs from the mutant's result. Return
    (killed mutants, survivors, mutation score %) — survivors point at missing
    tests.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(mutation_score, [
    (('age >= 18', ['age > 18', 'age <= 18', 'age >= 17', 'age >= 19', 'True'], [(30, True), (5, False)]), (['age <= 18', 'True'], ['age > 18', 'age >= 17', 'age >= 19'], 40.0)),
    (('age >= 18', ['age > 18', 'age <= 18', 'age >= 17', 'age >= 19', 'True'], [(30, True), (5, False), (18, True), (17, False)]), (['age > 18', 'age <= 18', 'age >= 17', 'age >= 19', 'True'], [], 100.0)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Evaluate every mutant on every test with eval.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Same, with any() and a compiled expression.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Evaluate every mutant on every test with eval.

```python
def mutation_score(original, mutants, tests):
    killed, survived = [], []
    for m in mutants:
        caught = False
        for age, want in tests:
            if eval(m, {"age": age}) != want:
                caught = True
        (killed if caught else survived).append(m)
    return (killed, survived, round(len(killed) / len(mutants) * 100, 1))
```

**Cost:** O(m · t) time, O(m) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Same, with any() and a compiled expression.

```python
def mutation_score(original, mutants, tests):
    code = {m: compile(m, "<mutant>", "eval") for m in mutants}
    killed = [m for m in mutants if any(eval(code[m], {"age": a}) != w for a, w in tests)]
    return (killed, [m for m in mutants if m not in killed], round(100 * len(killed) / len(mutants), 1))
```

**Cost:** O(m · t) time, O(m) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Evaluate every mutant on every test with eval. That costs O(m · t) time and O(m) space."

"I can do better: Same, with any() and a compiled expression. That's O(m · t) time and O(m) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/testing/mutation-testing-are-your-tests-any-good)

<a id="q16"></a>
### Q16. Run Only the Affected Tests

🟡 Interview standard · *Fast feedback in big repos*

`imports` maps each module (and test file) to the modules it imports. Given the changed modules, return the test files (names starting with `test_`) that import a changed module directly or indirectly, sorted.

**Examples**

- `imports = {'test_api.py': ['api', 'auth'], 'test_models.py': ['models'], 'test_utils.py': ['utils'], 'api': ['models', 'utils'], 'auth': ['utils'], 'models': ['db'], 'db': [], 'utils': [], 'cli': ['api']}, changed = ['db']` → `['test_api.py', 'test_models.py']`
- `imports = {'test_api.py': ['api', 'auth'], 'test_models.py': ['models'], 'test_utils.py': ['utils'], 'api': ['models', 'utils'], 'auth': ['utils'], 'models': ['db'], 'db': [], 'utils': [], 'cli': ['api']}, changed = ['auth']` → `['test_api.py']`

In [ ]:
def affected_tests(imports, changed):
    """
    imports maps each module (and test file) to the modules it imports. Given
    the changed modules, return the test files (names starting with test_) that
    import a changed module directly or indirectly, sorted.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(affected_tests, [
    (({'test_api.py': ['api', 'auth'], 'test_models.py': ['models'], 'test_utils.py': ['utils'], 'api': ['models', 'utils'], 'auth': ['utils'], 'models': ['db'], 'db': [], 'utils': [], 'cli': ['api']}, ['db']), ['test_api.py', 'test_models.py']),
    (({'test_api.py': ['api', 'auth'], 'test_models.py': ['models'], 'test_utils.py': ['utils'], 'api': ['models', 'utils'], 'auth': ['utils'], 'models': ['db'], 'db': [], 'utils': [], 'cli': ['api']}, ['auth']), ['test_api.py']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

For each test, DFS through its imports.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Reverse the graph once and BFS from the changes.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** For each test, DFS through its imports.

```python
def affected_tests(imports, changed):
    out = []
    for t in imports:
        if not t.startswith("test_"):
            continue
        seen, stack = set(), list(imports[t])
        while stack:
            m = stack.pop()
            if m not in seen:
                seen.add(m)
                stack += imports.get(m, [])
        if seen & set(changed):
            out.append(t)
    return sorted(out)
```

**Cost:** O(t · (n + e)) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Reverse the graph once and BFS from the changes.

```python
def affected_tests(imports, changed):
    rev = defaultdict(list)
    for m, deps in imports.items():
        for d in deps:
            rev[d].append(m)
    seen, stack = set(changed), list(changed)
    while stack:
        for up in rev[stack.pop()]:
            if up not in seen:
                seen.add(up)
                stack.append(up)
    return sorted(m for m in seen if m.startswith("test_"))
```

**Cost:** O(n + e) time, O(n + e) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: For each test, DFS through its imports. That costs O(t · (n + e)) time and O(n) space."

"I can do better: Reverse the graph once and BFS from the changes. That's O(n + e) time and O(n + e) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/testing/run-only-the-affected-tests)

<a id="q17"></a>
### Q17. Snapshot Testing

🟡 Interview standard · *Catch unexpected output changes*

Compare the current output with the stored snapshot (nested dicts) and return sorted differences as `(path, kind)` where path is dot-separated and kind is `'added'`, `'removed'` or `'changed'`.

**Examples**

- `old = {'user': {'id': 7, 'name': 'Asha', 'roles': ['admin']}, 'plan': 'pro', 'limits': {'api': 1000}}, new = {'user': {'id': 7, 'name': 'Asha K', 'roles': ['admin'], 'email': 'a@x.com'}, 'plan': 'pro', 'limits': {}}` → `[('limits.api', 'removed'), ('user.email', 'added'), ('user.name', 'changed')]`

In [ ]:
def snapshot_diff(old, new):
    """
    Compare the current output with the stored snapshot (nested dicts) and
    return sorted differences as (path, kind) where path is dot-separated and
    kind is 'added', 'removed' or 'changed'.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(snapshot_diff, [
    (({'user': {'id': 7, 'name': 'Asha', 'roles': ['admin']}, 'plan': 'pro', 'limits': {'api': 1000}}, {'user': {'id': 7, 'name': 'Asha K', 'roles': ['admin'], 'email': 'a@x.com'}, 'plan': 'pro', 'limits': {}}), [('limits.api', 'removed'), ('user.email', 'added'), ('user.name', 'changed')]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Recursive comparison building a list.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Recursion over the union of keys.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Recursive comparison building a list.

```python
def snapshot_diff(old, new):
    out = []
    def walk(a, b, prefix):
        for k in a:
            p = prefix + k
            if k not in b:
                out.append((p, "removed"))
            elif isinstance(a[k], dict) and isinstance(b[k], dict):
                walk(a[k], b[k], p + ".")
            elif a[k] != b[k]:
                out.append((p, "changed"))
        for k in b:
            if k not in a:
                out.append((prefix + k, "added"))
    walk(old, new, "")
    return sorted(out)
```

**Cost:** O(n) time, O(d) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Recursion over the union of keys.

```python
def snapshot_diff(old, new):
    def walk(a, b, pre):
        for k in a.keys() | b.keys():
            p = pre + k
            if k not in b:
                yield (p, "removed")
            elif k not in a:
                yield (p, "added")
            elif isinstance(a[k], dict) and isinstance(b[k], dict):
                yield from walk(a[k], b[k], p + ".")
            elif a[k] != b[k]:
                yield (p, "changed")
    return sorted(walk(old, new, ""))
```

**Cost:** O(n) time, O(d) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Recursive comparison building a list. That costs O(n) time and O(d) space."

"I can do better: Recursion over the union of keys. That's O(n) time and O(d) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `old = {'a': 1}, new = {'a': 2}`

1. walk() returns ('a', 'changed')
2. walk() returns None
3. snapshot\_diff() returns [('a', 'changed')]

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/testing/snapshot-testing)

<a id="q18"></a>
### Q18. Spot Test Smells

🟡 Interview standard · *Read tests with the ast module*

Parse each test's source with `ast` and report, per test function, the smells: `'no assert'` (no `assert` statement and no `pytest.raises`), `'sleeps'` (calls `time.sleep` or `sleep`), `'prints'` (calls `print`). Return `(test name, sorted smells)` only for tests with smells.

**Examples**

- `sources = ['def test_total():\n    assert total([1, 2]) == 3\n', 'def test_slow_job():\n    start_job()\n    time.sleep(5)\n    print(status())\n', "def test_raises():\n    with pytest.raises(ValueError):\n        parse('x')\n", 'def test_nothing():\n    result = compute()\n']` → `[('test_slow_job', ['no assert', 'prints', 'sleeps']), ('test_nothing', ['no assert'])]`

In [ ]:
import ast

def test_smells(sources):
    """
    Parse each test's source with ast and report, per test function, the smells:
    'no assert' (no assert statement and no pytest.raises), 'sleeps' (calls
    time.sleep or sleep), 'prints' (calls print). Return (test name, sorted
    smells) only for tests with smells.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(test_smells, [
    ((['def test_total():\n    assert total([1, 2]) == 3\n', 'def test_slow_job():\n    start_job()\n    time.sleep(5)\n    print(status())\n', "def test_raises():\n    with pytest.raises(ValueError):\n        parse('x')\n", 'def test_nothing():\n    result = compute()\n'],), [('test_slow_job', ['no assert', 'prints', 'sleeps']), ('test_nothing', ['no assert'])]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Walk the tree checking node types one by one.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Collect called names once with a comprehension.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Walk the tree checking node types one by one.

```python
def called_name(node):
    f = node.func
    return f.attr if isinstance(f, ast.Attribute) else getattr(f, "id", "")


def test_smells(sources):
    out = []
    for src in sources:
        fn = ast.parse(src).body[0]
        has_assert = has_raises = sleeps = prints = False
        for node in ast.walk(fn):
            if isinstance(node, ast.Assert):
                has_assert = True
            if isinstance(node, ast.Call):
                name = called_name(node)
                if name == "raises":
                    has_raises = True
                if name == "sleep":
                    sleeps = True
                if name == "print":
                    prints = True
        smells = []
        if not (has_assert or has_raises):
            smells.append("no assert")
        if prints:
            smells.append("prints")
        if sleeps:
            smells.append("sleeps")
        if smells:
            out.append((fn.name, smells))
    return out
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Collect called names once with a comprehension.

```python
def called_name(node):
    f = node.func
    return f.attr if isinstance(f, ast.Attribute) else getattr(f, "id", "")


def test_smells(sources):
    out = []
    for src in sources:
        fn = ast.parse(src).body[0]
        nodes = list(ast.walk(fn))
        calls = {called_name(n) for n in nodes if isinstance(n, ast.Call)}
        checks = {"no assert": not any(isinstance(n, ast.Assert) for n in nodes) and "raises" not in calls,
                  "sleeps": "sleep" in calls, "prints": "print" in calls}
        smells = sorted(k for k, bad in checks.items() if bad)
        if smells:
            out.append((fn.name, smells))
    return out
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Walk the tree checking node types one by one. That costs O(n) time and O(n) space."

"I can do better: Collect called names once with a comprehension. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/testing/spot-test-smells)

<a id="q19"></a>
### Q19. Should You Retry Flaky Tests?

🟡 Interview standard · *Odds of a red build*

Each of `n` flaky tests fails at random with probability `p` per run. Return `(chance the build fails with no retries, chance it fails when each test may be retried up to r times)`, as % rounded to 2. Retries hide flakiness — fix the root cause, but they buy time.

**Examples**

- `n = 20, p = 0.02, r = 2` → `(33.24, 0.02)`
- `n = 200, p = 0.01, r = 1` → `(86.6, 1.98)`

In [ ]:
def flaky_odds(n, p, r):
    """
    Each of n flaky tests fails at random with probability p per run. Return
    (chance the build fails with no retries, chance it fails when each test may
    be retried up to r times), as % rounded to 2. Retries hide flakiness — fix
    the root cause, but they buy time.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(flaky_odds, [
    ((20, 0.02, 2), (33.24, 0.02)),
    ((200, 0.01, 1), (86.6, 1.98)),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Loop multiplying pass chances.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Closed form with powers.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Loop multiplying pass chances.

```python
def flaky_odds(n, p, r):
    pass_once, pass_retry = 1.0, 1.0
    for _ in range(n):
        pass_once *= 1 - p
        pass_retry *= 1 - p ** (r + 1)
    return (round((1 - pass_once) * 100, 2), round((1 - pass_retry) * 100, 2))
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Closed form with powers.

```python
def flaky_odds(n, p, r):
    return (round(100 * (1 - (1 - p) ** n), 2), round(100 * (1 - (1 - p ** (r + 1)) ** n), 2))
```

**Cost:** O(1) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Loop multiplying pass chances. That costs O(n) time and O(1) space."

"I can do better: Closed form with powers. That's O(1) time and O(1) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/testing/should-you-retry-flaky-tests)

<a id="q20"></a>
### Q20. Shrink a Failing Example

🔴 Stretch · *Property-based testing (Hypothesis)*

Hypothesis finds a random input that breaks a property, then **shrinks** it to the simplest failing input. Starting from `example` (which fails `fails_property`), repeat until nothing changes: try deleting each element (first to last); keep a deletion if the result still fails. Then try replacing each element with half its value (integer division), keeping it if it still fails. Return the shrunk list.

**Examples**

- `example = [7, 3, 12, 9, 1, 30, 4]` → `[15, 0]`
- `example = [50, 50]` → `[6, 6]`

In [ ]:
def shrink(example):
    """
    Hypothesis finds a random input that breaks a property, then shrinks it to
    the simplest failing input. Starting from example (which fails
    fails_property), repeat until nothing changes: try deleting each element
    (first to last); keep a deletion if the result still fails. Then try
    replacing each element with half its value (integer division), keeping it if
    it still fails. Return the shrunk list.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(shrink, [
    (([7, 3, 12, 9, 1, 30, 4],), [15, 0]),
    (([50, 50],), [6, 6]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Rebuild lists by hand in nested loops.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

The same passes, with slicing.

</details>

<details>
<summary><b>✅ Answer: Step by step — plain Python</b></summary>

**Idea:** Rebuild lists by hand in nested loops.

```python
def fails_property(xs):
    # The bug: a function that breaks when the list has 2+ items adding up to more than 10.
    return len(xs) >= 2 and sum(xs) > 10


def shrink(example):
    xs = list(example)
    changed = True
    while changed:
        changed = False
        i = 0
        while i < len(xs):
            trial = xs[:i] + xs[i + 1:]
            if fails_property(trial):
                xs = trial
                changed = True
            else:
                i += 1
        for i in range(len(xs)):
            trial = list(xs)
            trial[i] = xs[i] // 2
            if trial[i] != xs[i] and fails_property(trial):
                xs = trial
                changed = True
    return xs
```

**Cost:** O(n² · v) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** The same passes, with slicing.

```python
def fails_property(xs):
    # The bug: a function that breaks when the list has 2+ items adding up to more than 10.
    return len(xs) >= 2 and sum(xs) > 10


def shrink(example):
    xs, changed = list(example), True
    while changed:
        changed, i = False, 0
        while i < len(xs):
            if fails_property(xs[:i] + xs[i + 1:]):
                xs, changed = xs[:i] + xs[i + 1:], True
            else:
                i += 1
        for i, v in enumerate(xs):
            t = xs[:i] + [v // 2] + xs[i + 1:]
            if v // 2 != v and fails_property(t):
                xs, changed = t, True
    return xs
```

**Cost:** O(n² · v) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Rebuild lists by hand in nested loops. That costs O(n² · v) time and O(n) space."

"I can do better: The same passes, with slicing. That's O(n² · v) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/engineering-toolbox/#/q/testing/shrink-a-failing-example)

---
# Part 4: Interview questions, out loud

Cover the answer, **say yours out loud**, then open it and compare. Aim for two or three clear sentences, then an example.

<details>
<summary><b>1. What makes a good unit test?</b></summary>

Fast, independent, repeatable, tests one behaviour, has a clear name that says what should happen, and fails with a message that explains why.

</details>

<details>
<summary><b>2. When should you mock?</b></summary>

At boundaries you don't control or that are slow or non-deterministic: network, payment providers, email, time, randomness. Don't mock the code you're testing, and prefer fakes for complex dependencies.

</details>

<details>
<summary><b>3. Is 100% coverage worth it?</b></summary>

Not as a goal. Coverage finds untested code; it doesn't prove the tests check the right things. Aim high on critical logic and use mutation testing to judge test quality.

</details>

<details>
<summary><b>4. How do you deal with flaky tests?</b></summary>

Detect them from run history, quarantine so they don't block merges, then fix the root cause: shared state, test order, real time, network or races.

</details>

<details>
<summary><b>5. What is TDD?</b></summary>

Test-driven development: write a failing test, write the simplest code to pass it, then refactor. It forces clear interfaces and gives you tests for free.

</details>

<details>
<summary><b>6. How do you test code that calls an external API?</b></summary>

Unit tests with a stub or mock of the client; a few integration tests against a sandbox or a recorded response (e.g. VCR); and contract tests so you notice when the API changes.

</details>


---
# Check yourself before moving on

You're ready for the next notebook when you can tick these **without looking**:

- [ ] I can explain: Your first pytest tests
- [ ] I can explain: Fixtures, mocks and testable code
- [ ] I can explain: How good are your tests?
- [ ] I can explain: Keeping a test suite fast and trustworthy

**Revision drill.** Come back in a few days, set a 25-minute timer per question, and solve these from a blank cell:

- [ ] [Q16](#q16) Run Only the Affected Tests
- [ ] [Q17](#q17) Snapshot Testing
- [ ] [Q18](#q18) Spot Test Smells
- [ ] [Q19](#q19) Should You Retry Flaky Tests?
- [ ] [Q20](#q20) Shrink a Failing Example

Spaced repetition beats re-reading: redoing a question after you've partly forgotten it is what makes it stick.